# 02 — GNN Anomaly Detection & Forward Probe
**cycle_project / SUBSTRATE** · Phase 2 analysis

Covers:
1. GNN training recap (loss curve)
2. Anomaly score timeline — all 150 windows
3. Proxy z-score heatmap per anomaly window
4. Top-8 anomaly windows vs known geophysical events
5. Forward probe summary (spectral → decay → fingerprint)
6. SUBSTRATE cross-instrument correlation

> **No GPU needed.** Loads pre-computed scores from `data/processed/`.  
> Re-train: set `RETRAIN = True` (requires torch + torch_geometric).


In [1]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC   = ROOT / "src"
PROC  = ROOT / "data" / "processed"
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(ROOT))  # substrate package lives at ROOT/substrate/

# Optional torch
try:
    import torch
    _TORCH_OK = True
except ImportError:
    _TORCH_OK = False

RETRAIN = False   # set True to retrain GNN (requires torch + torch_geometric)

# ── Dark theme ────────────────────────────────────────────────────────────────
BG   = "#0A0E1A"
BG2  = "#111827"
FG   = "#E2E8F0"
ACC1 = "#60A5FA"   # blue
ACC2 = "#F97316"   # orange
ACC3 = "#34D399"   # green
ACC4 = "#A78BFA"   # purple
RED  = "#F87171"   # red

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG2,
    "axes.edgecolor": "#334155", "axes.labelcolor": FG,
    "xtick.color": FG, "ytick.color": FG,
    "text.color": FG, "grid.color": "#1E293B",
    "grid.alpha": 0.6, "lines.linewidth": 1.5,
    "font.family": "monospace",
})

print(f"ROOT  : {ROOT}")
print(f"PROC  : {PROC}")
print(f"torch : {_TORCH_OK}  |  RETRAIN : {RETRAIN}")


ROOT  : /sessions/eloquent-inspiring-pasteur/mnt/cycle_project
PROC  : /sessions/eloquent-inspiring-pasteur/mnt/cycle_project/data/processed
torch : False  |  RETRAIN : False


In [2]:
EVENTS = {
    "Younger Dryas"   : {"age_bp": 12_900, "color": ACC1, "marker": "v"},
    "8.2 ka event"    : {"age_bp":  8_200, "color": ACC3, "marker": "^"},
    "LGM"             : {"age_bp": 20_000, "color": ACC4, "marker": "s"},
    "Laschamp"        : {"age_bp": 41_000, "color": RED,  "marker": "D"},
    "Mono Lake"       : {"age_bp": 34_000, "color": ACC2, "marker": "P"},
}

PROXY_COLS  = ["gisp2_d18o_norm","vostok_deuterium_norm","vostok_co2_norm",
               "grip_be10_norm","sint2000_norm"]
PROXY_SHORT = ["GISP2 δ¹⁸O","Vostok ΔTs","Vostok CO₂","GRIP ¹⁰Be","VADM"]

print("Known events:")
for name, ev in EVENTS.items():
    print(f"  {name:20s}  {ev['age_bp']:>7,} yr BP")


Known events:
  Younger Dryas          12,900 yr BP
  8.2 ka event            8,200 yr BP
  LGM                    20,000 yr BP
  Laschamp               41,000 yr BP
  Mono Lake              34,000 yr BP


In [3]:
df_proxy = pd.read_parquet(PROC / "aligned.parquet").sort_values("age_bp").reset_index(drop=True)
df_scores = pd.read_parquet(PROC / "anomaly_scores.parquet").sort_values("age_bp").reset_index(drop=True)

with open(PROC / "anomaly_scores.json") as f:
    scores_meta = json.load(f)

threshold = scores_meta["threshold"]
anomalies = pd.DataFrame(scores_meta["anomalies"])

print(f"Proxy grid  : {len(df_proxy):>5} rows  {df_proxy['age_bp'].min():.0f}–{df_proxy['age_bp'].max():.0f} yr BP")
print(f"Score windows: {len(df_scores):>4} rows  {df_scores['age_bp'].min():.0f}–{df_scores['age_bp'].max():.0f} yr BP")
print(f"Threshold    : {threshold:.6f}")
print(f"Anomaly hits : {len(anomalies)} windows above threshold")
print()
print("Anomaly windows:")
print(anomalies.to_string(index=False))


Proxy grid  :   801 rows  0–80000 yr BP
Score windows:  150 rows  3000–77500 yr BP
Threshold    : 0.029194
Anomaly hits : 8 windows above threshold

Anomaly windows:
 age_bp    score  rank
  32000 0.033751     1
  34500 0.032760     2
   7000 0.032341     3
  44500 0.031380     4
  70000 0.030427     5
  55500 0.030046     6
  41500 0.029745     7
  68500 0.029688     8


In [4]:
loss_png = PROC / "training_loss.png"

fig, ax = plt.subplots(1, 1, figsize=(11, 4), facecolor=BG)
ax.set_facecolor(BG2)

if loss_png.exists():
    img = plt.imread(str(loss_png))
    ax.imshow(img, aspect="auto")
    ax.axis("off")
    ax.set_title("GNN Training Loss (GraphAutoEncoder — pre-computed)", color=FG, fontsize=12)
else:
    ax.text(0.5, 0.5, "training_loss.png not found\nRun with RETRAIN=True on GPU machine",
            ha="center", va="center", color=FG, fontsize=13, transform=ax.transAxes)
    ax.axis("off")

plt.tight_layout()
plt.savefig(PROC / "nb02_training_loss.png", dpi=120, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved → nb02_training_loss.png")


Saved → nb02_training_loss.png


In [5]:
age_ka    = df_scores["age_bp"].values / 1000
scores    = df_scores["anomaly_score"].values
anom_mask = scores >= threshold

fig, ax = plt.subplots(figsize=(14, 5), facecolor=BG)
ax.set_facecolor(BG2)

# Filled area under curve
ax.fill_between(age_ka, scores, alpha=0.18, color=ACC1)

# Non-anomalous windows
ax.plot(age_ka[~anom_mask], scores[~anom_mask], "o",
        ms=3.5, color=ACC1, alpha=0.5, label="Normal")

# Anomalous windows
ax.plot(age_ka[anom_mask], scores[anom_mask], "o",
        ms=6, color=RED, alpha=0.9, zorder=5, label=f"Anomaly (≥{threshold:.4f})")

# Threshold line
ax.axhline(threshold, color=RED, ls="--", lw=1.2, alpha=0.8, label="Threshold (2σ)")

# Known events
for name, ev in EVENTS.items():
    ka = ev["age_bp"] / 1000
    if df_scores["age_bp"].min()/1000 <= ka <= df_scores["age_bp"].max()/1000:
        ax.axvline(ka, color=ev["color"], ls=":", lw=1.0, alpha=0.7)
        ax.text(ka + 0.3, ax.get_ylim()[1] * 0.93, name,
                color=ev["color"], fontsize=7.5, rotation=90, va="top")

ax.set_xlabel("Age (ka BP)", color=FG, fontsize=11)
ax.set_ylabel("Reconstruction Error", color=FG, fontsize=11)
ax.set_title("GNN Anomaly Score — All 500-yr Windows", color=FG, fontsize=13, fontweight="bold")
ax.invert_xaxis()
ax.legend(fontsize=9, facecolor=BG, labelcolor=FG, framealpha=0.8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PROC / "nb02_anomaly_timeline.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Anomalous windows: {anom_mask.sum()} / {len(scores)}")


Anomalous windows: 8 / 150


In [6]:
# Build proxy matrix for scored windows (merge on age_bp)
df_merged = df_scores.merge(df_proxy[["age_bp"] + PROXY_COLS], on="age_bp", how="left")
df_merged = df_merged.sort_values("age_bp", ascending=False)  # oldest first → top of heatmap

Z = df_merged[PROXY_COLS].values.T   # shape (5, n_windows)
ages_hm = df_merged["age_bp"].values / 1000
sc_hm    = df_merged["anomaly_score"].values

fig = plt.figure(figsize=(16, 7), facecolor=BG)
gs  = gridspec.GridSpec(2, 1, height_ratios=[3.5, 1], hspace=0.08)

# ── Top: proxy heatmap ────────────────────────────────────────────────────────
ax_hm = fig.add_subplot(gs[0])
norm  = TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)
im    = ax_hm.imshow(Z, aspect="auto", cmap="RdBu_r", norm=norm,
                     extent=[ages_hm[-1], ages_hm[0], -0.5, 4.5])

ax_hm.set_yticks(range(5))
ax_hm.set_yticklabels(PROXY_SHORT[::-1], fontsize=10, color=FG)
ax_hm.set_xticks([])
ax_hm.set_title("Proxy Z-Score Heatmap × GNN Windows", color=FG, fontsize=13, fontweight="bold")

cb = plt.colorbar(im, ax=ax_hm, orientation="vertical", pad=0.01, shrink=0.9)
cb.set_label("z-score", color=FG, fontsize=9)
cb.ax.yaxis.set_tick_params(color=FG)
plt.setp(cb.ax.yaxis.get_ticklabels(), color=FG)

for name, ev in EVENTS.items():
    ka = ev["age_bp"] / 1000
    ax_hm.axvline(ka, color=ev["color"], ls=":", lw=1.1, alpha=0.85)

# ── Bottom: anomaly score bar ─────────────────────────────────────────────────
ax_sc = fig.add_subplot(gs[1])
colors_bar = [RED if s >= threshold else ACC1 for s in sc_hm]
ax_sc.bar(ages_hm, sc_hm, width=np.diff(ages_hm).mean() * 0.9 if len(ages_hm) > 1 else 0.5,
          color=colors_bar, alpha=0.85)
ax_sc.axhline(threshold, color=RED, ls="--", lw=1.0, alpha=0.8)
ax_sc.set_xlabel("Age (ka BP)", color=FG, fontsize=11)
ax_sc.set_ylabel("Score", color=FG, fontsize=9)
ax_sc.invert_xaxis()
ax_sc.grid(True, alpha=0.25)

for name, ev in EVENTS.items():
    ka = ev["age_bp"] / 1000
    if ages_hm.min() <= ka <= ages_hm.max():
        ax_sc.axvline(ka, color=ev["color"], ls=":", lw=1.0, alpha=0.7)

# Legend
patches = [mpatches.Patch(color=v["color"], label=k) for k,v in EVENTS.items()]
ax_hm.legend(handles=patches, fontsize=8, facecolor=BG, labelcolor=FG,
             loc="upper right", framealpha=0.85)

plt.savefig(PROC / "nb02_proxy_heatmap.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved → nb02_proxy_heatmap.png")


Saved → nb02_proxy_heatmap.png


In [7]:
# For each anomaly window, show the proxy fingerprint (bar chart)
top_n = min(8, len(anomalies))
top   = anomalies.nlargest(top_n, "score").reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(16, 7), facecolor=BG)
axes = axes.flatten()

for i, row in top.iterrows():
    ax = axes[i]
    ax.set_facecolor(BG2)

    age_bp = row["age_bp"]
    score  = row["score"]

    # Get proxy values for this window's age
    win_row = df_merged[df_merged["age_bp"] == age_bp]
    if win_row.empty:
        # nearest
        idx = (df_merged["age_bp"] - age_bp).abs().idxmin()
        win_row = df_merged.iloc[[idx]]

    vals = win_row[PROXY_COLS].values.flatten()

    bar_colors = [RED if v < -1.5 else ACC2 if v > 1.5 else ACC1 for v in vals]
    bars = ax.barh(range(5), vals, color=bar_colors, alpha=0.85)
    ax.axvline(0, color=FG, lw=0.8, alpha=0.5)
    ax.axvline(-2, color=RED, lw=0.6, ls="--", alpha=0.4)
    ax.axvline(+2, color=ACC2, lw=0.6, ls="--", alpha=0.4)

    ax.set_yticks(range(5))
    ax.set_yticklabels(PROXY_SHORT, fontsize=8, color=FG)
    ax.set_xlim(-4, 4)
    ax.set_xlabel("z-score", color=FG, fontsize=8)

    # Match to known events
    match = ""
    for ev_name, ev in EVENTS.items():
        if abs(age_bp - ev["age_bp"]) < 3000:
            match = f" ← {ev_name}"
            break

    ax.set_title(f"#{row['rank']}  {age_bp/1000:.1f} ka BP  (score={score:.5f}){match}",
                 color=FG, fontsize=8.5, fontweight="bold")
    ax.grid(True, axis="x", alpha=0.25)

# Hide unused panels
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Top Anomaly Windows — Proxy Fingerprints", color=FG, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PROC / "nb02_top_anomalies.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved → nb02_top_anomalies.png")


Saved → nb02_top_anomalies.png


In [8]:
print("=" * 65)
print(f"{'Event':<22} {'Event BP':>9} {'Nearest window':>14} {'Score':>9} {'Δ ka':>6} {'Flag':>8}")
print("-" * 65)

for ev_name, ev in EVENTS.items():
    ev_bp = ev["age_bp"]
    if df_scores["age_bp"].min() <= ev_bp <= df_scores["age_bp"].max():
        idx    = (df_scores["age_bp"] - ev_bp).abs().idxmin()
        near   = df_scores.iloc[idx]
        delta  = (near["age_bp"] - ev_bp) / 1000
        sc     = near["anomaly_score"]
        flag   = "ANOMALY" if sc >= threshold else "normal"
        print(f"{ev_name:<22} {ev_bp:>9,} {near['age_bp']:>14,} {sc:>9.5f} {delta:>+6.1f} {flag:>8}")
    else:
        print(f"{ev_name:<22} {ev_bp:>9,} {'(out of range)':>14}")
print("=" * 65)
print(f"\nThreshold : {threshold:.5f}")
print(f"Windows above threshold: {(df_scores['anomaly_score'] >= threshold).sum()}")


Event                   Event BP Nearest window     Score   Δ ka     Flag
-----------------------------------------------------------------
Younger Dryas             12,900       13,000.0   0.02519   +0.1   normal
8.2 ka event               8,200        8,000.0   0.01663   -0.2   normal
LGM                       20,000       20,000.0   0.01812   +0.0   normal
Laschamp                  41,000       41,000.0   0.02136   +0.0   normal
Mono Lake                 34,000       34,000.0   0.02124   +0.0   normal

Threshold : 0.02919
Windows above threshold: 8


In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

probe_json = PROC / "probe_state.json"
if probe_json.exists():
    with open(probe_json) as f:
        probe = json.load(f)
    print("Forward Probe State:")
    for k, v in probe.items():
        print(f"  {k}: {v}")
else:
    print("probe_state.json not found — running forward probe modules...")
    probe = {}

# Load and show pre-computed probe figures
fig_paths = {
    "Spectral Analysis" : PROC / "vadm_spectrum.png",
    "Decay Forecast"    : PROC / "vadm_forecast.png",
    "LSTM Ensemble"     : PROC / "vadm_lstm_forecast.png",
}

n = sum(1 for p in fig_paths.values() if p.exists())
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(7*n, 5), facecolor=BG)
    if n == 1:
        axes = [axes]
    j = 0
    for title, path in fig_paths.items():
        if path.exists():
            img = plt.imread(str(path))
            axes[j].imshow(img, aspect="auto")
            axes[j].axis("off")
            axes[j].set_title(title, color=FG, fontsize=11, fontweight="bold")
            j += 1
    plt.tight_layout()
    plt.savefig(PROC / "nb02_forward_probe_summary.png", dpi=110, bbox_inches="tight", facecolor=BG)
    plt.show()


Forward Probe State:
  pre_excursion_prob: 0.866667
  lstm_vadm_1kyr: 0.718639
  lstm_vadm_5kyr: 0.697694
  instrumental_threshold_yr: 2423
  generated_at: 2026-05-06T00:55:55.919130+00:00


In [10]:
# Run spectral + decay + fingerprint (no torch needed)
from forward_probe.spectral    import run_spectral
from forward_probe.decay_model import run_decay_model
from forward_probe.fingerprint import run_fingerprint

parquet_path = PROC / "aligned.parquet"
output_dir   = PROC

print("=" * 55)
print("SPECTRAL ANALYSIS")
print("=" * 55)
r_spec = run_spectral(parquet_path=parquet_path, output_dir=output_dir)
print(f"Top FFT periods  : {[int(x) for x in r_spec['dominant_periods'][:3]]} yr")
print(f"Top CWT periods  : {[int(x) for x in r_spec['cwt_dominant_periods'][:3]]} yr")

print()
print("=" * 55)
print("DECAY MODEL")
print("=" * 55)
r_decay = run_decay_model(parquet_path=parquet_path, output_dir=output_dir)
for model, info in r_decay["thresholds"].items():
    yr  = info.get("threshold_yr")
    lo  = info.get("ci_lo")
    hi  = info.get("ci_hi")
    if yr:
        ci = f"[{lo:,.0f}–{hi:,.0f}]" if lo and hi else "n/a"
        print(f"  {model:<14}: {yr:>8,.0f} yr  CI {ci}")
    else:
        print(f"  {model:<14}: field not decaying under this model")

print()
print("=" * 55)
print("EXCURSION FINGERPRINT")
print("=" * 55)
r_fp = run_fingerprint(parquet_path=parquet_path)
print(f"Excursion probability : {r_fp['probability']:.3f}")
print(f"LOO accuracy          : {r_fp['loo_accuracy']:.3f}")
print(f"Status                : {r_fp['status']}")
print(f"n_excursions          : {r_fp['n_excursions']}")


SPECTRAL ANALYSIS
[SPECTRUM] Loading VADM time series...
[SPECTRUM] 801 samples × 100 yr/sample = 80,100 yr span
[SPECTRUM] Running FFT...
[SPECTRUM] Running CWT (Ricker)...
[SPECTRUM] Top 5 FFT periods: ['26,700 yr', '10,012 yr', '8,010 yr', '5,721 yr', '3,483 yr']
[SPECTRUM] Top 5 CWT periods: ['30,212 yr']


[SPECTRUM] Saved → /sessions/eloquent-inspiring-pasteur/mnt/cycle_project/data/processed/vadm_spectrum.png
Top FFT periods  : [26700, 10012, 8010] yr
Top CWT periods  : [30211] yr

DECAY MODEL
[DECAY] Loading Sint-2000 data...
[DECAY] Fitting on 51 points (last 5,000 yr)
[DECAY] Under linear       decay: threshold in 4,181 ±3,516 years (95% CI)
[DECAY] Under exponential  decay: threshold in 7,539 ±4,857 years (95% CI)
[DECAY] Under power_law    decay: threshold in 238,813 ±406,675 years (95% CI)
[DECAY] Under instrumental decay: threshold in 2,423 ±969 years (95% CI)


[DECAY] Saved -> /sessions/eloquent-inspiring-pasteur/mnt/cycle_project/data/processed/vadm_forecast.png
  linear        :    4,181 yr  CI [2,119–9,152]
  exponential   :    7,539 yr  CI [4,482–14,196]
  power_law     :  238,813 yr  CI [28,303–841,654]
  instrumental  :    2,423 yr  CI [1,939–3,877]

EXCURSION FINGERPRINT
[FINGERPRINT] Loaded 7 excursion timestamps from anomaly_scores.json (1 Holocene anomalies filtered)
[FINGERPRINT] Loading proxy data...
[FINGERPRINT] Extracting pre-excursion windows...
[FINGERPRINT] 7 pre-excursion windows extracted
[FINGERPRINT] Sampling stable windows...
[FINGERPRINT] 17 stable windows extracted
[FINGERPRINT] Extracting current 3,000-yr window...
[FINGERPRINT] Training Random Forest classifier...


[FINGERPRINT] Running LOO cross-validation...


[FINGERPRINT] Pre-excursion probability: 0.210 (threshold 0.5) → STABLE
[FINGERPRINT] LOO accuracy: 3/7 = 0.43
Excursion probability : 0.210
LOO accuracy          : 0.429
Status                : STABLE
n_excursions          : 7


In [11]:
try:
    sys.path.insert(0, str(ROOT / "src"))
    from substrate import SubstrateLab
    from substrate.lab import SubstrateResult
    _SUBSTRATE_OK = True
except ImportError:
    _SUBSTRATE_OK = False
    print("substrate module not found — skipping SUBSTRATE cells")

if not _SUBSTRATE_OK:
    print("Install substrate or run from project root with src/ on PYTHONPATH")
else:
    lab = SubstrateLab(data_root=PROC / "substrate_cache", gpu=False)
    print("Available instruments:", lab.available)

    print()
    print("-- Geomagnetic anomaly scan -----------------")
    r_geo = lab.run("geomagnetic", task="anomaly_scan")
    print(f"  anomaly_windows: {len(r_geo.data['anomaly_windows'])} events")
    if r_geo.data["anomaly_windows"]:
        for w in r_geo.data["anomaly_windows"][:3]:
            print(f"    {w}")

    print()
    print("-- Mythology correlation --------------------")
    r_myth = lab.run("mythology", task="correlate_events")
    print(f"  table rows: {len(r_myth.data['table'])}")
    for row in r_myth.data["table"][:4]:
        print(f"    {row.get('kyr_bp','')} ka  {row.get('tradition','')}  {str(row.get('label',''))[:50]}")

    print()
    print("-- Cross-instrument correlation -------------")
    corr = lab.correlate([r_geo, r_myth], method="temporal_overlap")
    key  = "synchronous_events" if "synchronous_events" in corr.data else "sync_events"
    print(f"  synchronous events: {len(corr.data[key])}")
    for ev in corr.data[key][:5]:
        print(f"    {ev}")

    print()
    print("-- Markdown report --------------------------")
    rpt = lab.report(corr, fmt="markdown")
    print(rpt.data["text"][:600])


Available instruments: ['geomagnetic', 'forecast', 'simulation', 'mythology', 'coherence', 'quantum_bio']

-- Geomagnetic anomaly scan -----------------
  anomaly_windows: 0 events

-- Mythology correlation --------------------
  table rows: 4
    12.9 ka  STUB  Younger Dryas onset
    11.7 ka  STUB  YD termination
    41.0 ka  STUB  Laschamps excursion
    74.0 ka  STUB  Toba supervolcano

-- Cross-instrument correlation -------------
  synchronous events: 0

-- Markdown report --------------------------
# SUBSTRATE Report — `__correlator__` / `temporal_overlap`
*Generated: 2026-05-13T02:32:45Z*

No overlap data available.

## Provenance
```json
{
  "method": "temporal_overlap",
  "instruments": [
    "geomagnetic",
    "mythology"
  ],
  "warnings": [
    "Need \u22652 instruments with anomaly_windows for temporal overlap"
  ],
  "timestamp_utc": "2026-05-13T02:32:45Z"
}
```

## ⚠️ Warnings
- Need ≥2 instruments with anomaly_windows for temporal overlap


In [12]:
# Final 2×2 summary panel
fig, axes = plt.subplots(2, 2, figsize=(16, 12), facecolor=BG)

panels = [
    ("nb02_anomaly_timeline.png", "Anomaly Score Timeline"),
    ("nb02_proxy_heatmap.png",    "Proxy Z-Score Heatmap"),
    ("nb02_top_anomalies.png",    "Top Anomaly Windows"),
    ("vadm_forecast.png",         "VADM Decay Forecast"),
]

for ax, (fname, title) in zip(axes.flatten(), panels):
    path = PROC / fname
    if path.exists():
        img = plt.imread(str(path))
        ax.imshow(img, aspect="auto")
    else:
        ax.text(0.5, 0.5, f"{fname}\nnot found", ha="center", va="center",
                color=FG, transform=ax.transAxes)
    ax.axis("off")
    ax.set_title(title, color=FG, fontsize=11, fontweight="bold", pad=6)

fig.suptitle("cycle_project / SUBSTRATE — Phase 2 Summary",
             color=FG, fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
out = PROC / "nb02_summary_panel.png"
plt.savefig(out, dpi=130, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved → {out}")


Saved → /sessions/eloquent-inspiring-pasteur/mnt/cycle_project/data/processed/nb02_summary_panel.png


---
## Optional: Retrain GNN

Run the cell below only with `RETRAIN = True` and `torch` + `torch_geometric` installed (GPU machine).


In [13]:
if RETRAIN:
    if not _TORCH_OK:
        print("torch not available — set RETRAIN=False or install torch + torch_geometric")
    else:
        from cycle_detect.gnn_prototype import run_anomaly_scan
        result = run_anomaly_scan(
            data_root=ROOT / "data",
            epochs=120,
            threshold_sigma=2.0,
        )
        print("Retrain complete")
        print(f"  anomaly_windows : {len(result.get('anomaly_windows', []))}")
        print(f"  threshold       : {result.get('threshold', 'n/a'):.6f}")
else:
    print("RETRAIN=False — using pre-computed scores from data/processed/")
    print("Set RETRAIN=True and run on GPU machine to update.")


RETRAIN=False — using pre-computed scores from data/processed/
Set RETRAIN=True and run on GPU machine to update.
